# SODA: from instrument files to an upload-ready folder

This notebook turns the files your instrument (or analysis pipeline) wrote into the static
files SODA reads in the browser. You put each dataset's files in a fixed folder (Section 1),
and the notebook checks them, converts them and writes **one folder, `export/`**, that you
upload to the cloud as it is.

| Section | What happens | Output |
|---|---|---|
| 0. Setup | finds the repository, checks Python packages, sets your work folder | |
| 1. Put your files here | creates the input folders and says exactly which files go where | `inputs/` |
| 2. Check the inputs | lists every dataset found and anything missing | a table |
| 3. Prepare | reads each dataset once into a working copy | `cache/` (never upload) |
| 4. Export | cuts images into tiles, writes gene files and UMAPs | `export/` |
| 5. De-identification check | scans `export/` for sample IDs | must say `clean` |
| 6. Look before you upload | size per dataset, then open it on your own computer | |
| 7. Upload and link | `config.js` entries and the upload command for `export/` | |

Run the cells from top to bottom. Each step skips datasets that are already done, so you can
stop and start again. Commands are the same ones in `docs/PREPARING_DATA.md`.

## 0. Setup

Use a separate Python environment for SODA, because it needs older versions of `zarr` and
`tifffile` than many other tools. Once, from the repository folder (keep the path short on Windows):

```
python -m venv .venv
.venv\Scripts\activate              (macOS / Linux: source .venv/bin/activate)
pip install -r web/SpatialViewer/requirements.txt -r web/SpatialViewer/requirements-clustering.txt
pip install jupyter
```

Then open this notebook with that environment's Jupyter (or pick `.venv` as the kernel in VS Code).
The second requirements file is optional: without it the export skips the whole-section UMAP and clusters.

In [ ]:
import os, sys, json, shutil, subprocess, time, importlib
from pathlib import Path

def find_repo(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "web" / "SpatialViewer" / "prepare.py").exists():
            return p
    raise FileNotFoundError("Open this notebook from inside the SODA repository "
                            "(the folder that has web/SpatialViewer/prepare.py), or set REPO by hand.")

REPO = find_repo()
SV = REPO / "web" / "SpatialViewer"          # the scripts: prepare.py, deid.py, static_serve.py
PY = sys.executable                          # the same Python this notebook runs on
print("repository:", REPO)
print("python:    ", PY, sys.version.split()[0])

need = {"numpy": "core", "tifffile": "core", "zarr": "core", "imagecodecs": "core", "PIL": "core",
        "h5py": "core", "pyarrow": "core", "scipy": "core", "cryptography": "core",
        "sklearn": "UMAP (optional)", "umap": "UMAP (optional)", "igraph": "UMAP (optional)",
        "leidenalg": "UMAP (optional)"}
missing = []
for mod, what in need.items():
    try:
        m = importlib.import_module(mod)
        print(f"  ok       {mod:12s} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"  MISSING  {mod:12s} ({what})")
        missing.append(mod)
def _major(mod):
    try:
        return int(importlib.import_module(mod).__version__.split(".")[0])
    except Exception:
        return None
wrong = []
if (_major("zarr") or 0) >= 3:
    wrong.append("zarr must be 2.x (found 3+): images cannot be written")
if (_major("tifffile") or 0) >= 2025:
    wrong.append("tifffile must be older than 2025 (it needs zarr 3)")
for w in wrong:
    print("  WRONG VERSION:", w)
if wrong or any(need[m] == "core" for m in missing):
    print("\nThis Python is not set up for SODA. Make the .venv above, install the requirements in it,"
          "\nand run this notebook with that environment. Stop here until this cell shows no MISSING core or WRONG lines.")

### Settings: edit this cell

- **`WORK`** is one folder on a big, fast disk. Everything goes inside it:
  `inputs/` (your files), `cache/` (working copies, large, never uploaded) and
  `export/` (what you upload). Plan for about 2-3 times the size of your raw files.
- **`KEY_FILE`** is the de-identification key: it maps the numbers used in the export back to
  your real sample IDs. Keep it in a private folder **outside** `WORK` and outside the repository,
  and never upload or share it. See `docs/DEIDENTIFICATION.md`.

In [ ]:
WORK = Path(r"D:\SODA_work")                 # EDIT: a big disk; inputs/, cache/ and export/ go inside
KEY_FILE = Path(os.environ.get("SPATIALVIZ_KEY") or Path.home() / "private" / "soda" / "sample_id_key.csv")  # EDIT if you keep it elsewhere

TILE_FORMAT = "jpg"        # "jpg" is several times smaller; "png" is lossless (use it when exact intensities matter)
VISIUM_BINS = "008um,016um"  # Visium HD layers to keep. The 2 um layer is huge and mostly one transcript per bin
WORKERS = 8                # parallel tile writers during export
MARKERS = {                # marker-gene lists for suggested cluster names ("" = the generic list)
    "visium": "",          # e.g. str(SV / "markers" / "nerve.json")
    "proteomics": "",      # e.g. str(SV / "markers" / "akoya_nerve.json")
}

INPUTS, CACHE, EXPORT = WORK / "inputs", WORK / "cache", WORK / "export"
for d in (INPUTS, CACHE, EXPORT, KEY_FILE.parent):
    d.mkdir(parents=True, exist_ok=True)
os.environ["SPATIALVIZ_KEY"] = str(KEY_FILE)  # every script started from here uses this key
print("work folder:", WORK)
print("key file:   ", KEY_FILE, "(exists)" if KEY_FILE.exists() else "(will be created on first use)")
if KEY_FILE.resolve().is_relative_to(WORK.resolve()) or KEY_FILE.resolve().is_relative_to(REPO.resolve()):
    print("WARNING: keep the key file outside the work folder and the repository.")

## 1. Put your files here

Run the next cell once. It creates one folder per kind of data under `inputs/`:

```
inputs/
  proteomics_file/      one multichannel image per dataset (QPTIFF, OME-TIFF)
  proteomics_markers/   a folder of one TIFF per marker per dataset (Akoya-style export)
  visium/               Visium HD or classic Visium: Space Ranger outs/ + the full-size image
  xenium/               a Xenium output folder
  cosmx/                a CosMx SMI flat-file folder (one slide)
  merscope/             a MERSCOPE region folder
  scrnaseq/             single-cell / single-nucleus RNA-seq with a UMAP already made
```

**Each dataset is one subfolder, and the subfolder's name becomes the dataset id** (letters,
digits, `_` and `-`; no patient or sample identifiers). For example
`inputs/visium/nerve_section1/`. Exactly what goes inside, per instrument:

### Spatial proteomics: one file (Akoya PhenoCycler / CODEX `.qptiff`, any OME-TIFF)

```
inputs/proteomics_file/<dataset_id>/
    slide.qptiff            <- the scan: *.qptiff, *.ome.tif, *.ome.tiff or *.tif (exactly one file)
```
Channel names are read from the file. If they are missing, set `channels` in Section 2's options.

### Spatial proteomics: one TIFF per marker (Akoya / inForm component export)

```
inputs/proteomics_markers/<dataset_id>/
    01_DAPI_<sample>_gray.tif
    02_CD45_<sample>_gray.tif
    03_PanCK_<sample>_gray.tif      <- names like NN_Marker_<anything>_gray.tif, all the same size
    ...
```
`NN` sets the order and the marker name is the part after it. `_gray` files are preferred;
`_red` / `_green` / `_blue` / `_rgb` are used only when a marker has no grey file.
These files rarely store a pixel size, so give `pixel_size` (µm per pixel) in Section 2's options.

### Visium HD (Space Ranger 3+)

```
inputs/visium/<dataset_id>/
    image.tif                               <- the SAME full-resolution image you gave
                                               `spaceranger count --image` (H&E or IF; .tif/.tiff/.btf)
    outs/                                   <- Space Ranger's outs/ folder, unchanged
        binned_outputs/
            square_008um/
                filtered_feature_bc_matrix.h5
                spatial/tissue_positions.parquet
            square_016um/
                filtered_feature_bc_matrix.h5
                spatial/tissue_positions.parquet
            square_002um/ ...               (left out unless you add 002um to VISIUM_BINS)
```
Do not use `tissue_hires_image.png` as the image: it is downscaled and the bins will not line up.

**Your own UMAP and clusters (optional, Visium HD or classic Visium).** If you already clustered
this dataset in Seurat or scanpy, put the results next to `outs/` and SODA shows them (by name,
with your cell-type names and colours) next to its own clusters:

```
inputs/visium/<dataset_id>/
    my_umap.csv          <- barcode,UMAP_1,UMAP_2 (optional: without it your clusters go on SODA's UMAP)
    my_clusters.csv      <- barcode,seurat_clusters,cell_type,... (also my_clusters_*.csv; optional
                            colour column cell_type_color with #rrggbb)
    my_analysis.h5ad     <- OR an AnnData file instead: obs columns + obsm["X_umap"] (+ uns palettes)
```
They belong to **one** bin size: set `user_layer` (e.g. `"008um"`) for the dataset in Section 2's
options. How to write them from R or Python: `docs/PREPARING_DATA.md`, "Your own UMAP and clusters".

### Visium (classic, 55 µm spots)

```
inputs/visium/<dataset_id>/
    image.tif                               <- the full-resolution image given to spaceranger
    outs/
        filtered_feature_bc_matrix.h5
        spatial/tissue_positions.parquet    (or tissue_positions.csv / tissue_positions_list.csv)
```

### Xenium (Xenium Onboard Analysis 2.0+)

```
inputs/xenium/<dataset_id>/                 <- the whole Xenium output folder, as the instrument wrote it
    morphology_focus/
        ch0000_dapi.ome.tif                 <- USED: the stain images (names may be morphology_focus_0000.ome.tif)
        ch0001_atp1a1_cd45_e-cadherin.ome.tif
        ...
    Custom_annotation.geojson               <- USED if present: an overlay, in image pixels
    experiment.xenium                       <- USED: the pixel size
    cells.parquet  (or cells.csv.gz)        <- USED: cell centroids and areas
    cell_feature_matrix.h5  (or cell_feature_matrix/ with matrix.mtx.gz)  <- USED: counts per cell
    transcripts.parquet  (or transcripts.zarr.zip)  <- USED: single molecules, quality >= 20
    cell_boundaries.parquet, nucleus_boundaries.parquet  <- not used yet (cells are drawn as dots)
    cells.zarr.zip, analysis/, morphology.ome.tif         <- not needed
```
SODA shows the stains, the **cells** as an expression layer (gene maps, UMAP, clusters, QC,
circle -> UMAP and the volcano work on cells as on Visium bins) and the **molecules**: pick genes
in the Molecules panel to draw their transcripts as dots. Control probes and codewords are left out.
If the cell files or the transcripts are missing, that part is skipped and the rest is still prepared.
Reading the transcripts of a whole slide takes several minutes; set `"no_transcripts": True` in the
options (section 2) to skip them.

### CosMx SMI (flat files exported from AtoMx, or an older public release)

```
inputs/cosmx/<dataset_id>/                  <- one slide: the flat-file folder (files may sit up to two folders down)
    <slide>_exprMat_file.csv  (or .csv.gz)  <- USED: counts per cell (fov, cell_ID, one column per target)
    <slide>_metadata_file.csv               <- USED: cell centroids (CenterX/Y_local_px, CenterX/Y_global_px) and Area
    <slide>_fov_positions_file.csv          <- USED: where each FOV sits on the slide
    <slide>_tx_file.csv  (or .csv.gz)       <- USED: single molecules (target, x_global_px, y_global_px)
    Morphology2D/*_F001.TIF ...             <- USED if every FOV has one: the raw stain images (16-bit, one page per stain)
    CellComposite/CellComposite_F001.jpg ...  <- USED otherwise: the rendered colour picture of each FOV
    CellLabels/CellLabels_F001.tif ...      <- USED only to check which way up the FOV images are
    <slide>-polygons.csv, CompartmentLabels/, CellOverlay/   <- not used (cells are drawn as dots)
```
SODA stitches the FOV images into one picture using the FOV positions, and shows the **cells**
and the **molecules** as for Xenium. Negative probes (`NegPrb*`, `Negative*`), `SystemControl*`
and `FalseCode*` codes and the `cell_ID` 0 rows (transcripts in no cell) are left out. If a slide
has no images at all it still opens, on a grey transcript-density background. One folder = one
slide; give each slide of a run its own dataset. The pixel size is taken from the FOV size
(0.18 or 0.12 µm); set `pixel_size` in the options (section 2) if the log says it was assumed.

### MERSCOPE (Vizgen region folder)

```
inputs/merscope/<dataset_id>/               <- one region_N folder, as the instrument (or VPT) wrote it
    images/
        mosaic_DAPI_z3.tif                  <- USED: one z-plane of every stain (the middle one unless you set `z`)
        mosaic_PolyT_z3.tif
        mosaic_Cellbound1_z3.tif ...
        micron_to_mosaic_pixel_transform.csv  <- USED: places cells and molecules on the images
    cell_by_gene.csv                        <- USED: counts per cell
    cell_metadata.csv                       <- USED: cell centroids (center_x, center_y) and bounding boxes
    detected_transcripts.csv  (or .csv.gz / .parquet)  <- USED: single molecules (gene, global_x, global_y)
    cell_boundaries.parquet, cell_boundaries/   <- not used (cells are drawn as dots)
```
The other z-planes are not needed: copy only the plane you want if disk space is short (each
mosaic is several GB). `Blank-*` barcodes are left out. The two cell tables may sit one folder
down (for example a `vpt-run/` folder from the Vizgen Post-processing Tool). Without the
`images/` folder the dataset still opens, on a grey transcript-density background.

### Single-cell / single-nucleus RNA-seq (UMAP already computed)

Either an AnnData file:
```
inputs/scrnaseq/<dataset_id>/
    cells.h5ad          <- with the UMAP in obsm["X_umap"] and clusters in obs (e.g. "leiden")
```
or Cell Ranger output:
```
inputs/scrnaseq/<dataset_id>/
    filtered_feature_bc_matrix/             <- matrix.mtx(.gz), features.tsv(.gz), barcodes.tsv(.gz)
    analysis/umap/gene_expression_2_components/projection.csv          <- barcode,UMAP1,UMAP2
    analysis/clustering/gene_expression_graphclust/clusters.csv        <- optional, colour-by
```
(Instead of `analysis/...`, a `umap.csv` with columns `barcode,UMAP1,UMAP2` next to the matrix
folder works too; any other `*.csv` beside it with a `barcode` column becomes extra colour-by columns.)

**Big files you don't want to copy:** leave them where they are and point the dataset at them
with `src` in Section 2's options.

In [ ]:
KINDS = {
    "proteomics_file":    "ONE multichannel image: *.qptiff, *.ome.tif, *.ome.tiff or *.tif",
    "proteomics_markers": "one TIFF per marker: NN_Marker_<anything>_gray.tif, all the same size",
    "visium":             "image.tif (the image given to spaceranger --image) + Space Ranger outs/",
    "xenium":             "the whole Xenium output folder (stains, cells, transcripts)",
    "cosmx":              "a CosMx flat-file folder: *_exprMat_file, *_metadata_file, *_fov_positions_file, *_tx_file (+ image folders)",
    "merscope":           "a MERSCOPE region folder: images/, cell_by_gene.csv, cell_metadata.csv, detected_transcripts.csv",
    "scrnaseq":           "cells.h5ad with obsm['X_umap'], OR filtered_feature_bc_matrix/ + a UMAP csv",
}
for kind, what in KINDS.items():
    d = INPUTS / kind
    d.mkdir(parents=True, exist_ok=True)
    readme = d / "README.txt"
    if not readme.exists():
        readme.write_text(f"Put one subfolder per dataset here. The subfolder name is the dataset id.\n"
                          f"Inside it: {what}\nFull layout: Section 1 of SODA_pipeline.ipynb.\n", encoding="utf-8")
    print(f"{str(d):60s} {what}")

## 2. Check the inputs

**Options per dataset (optional).** Keyed by dataset id. Leave a dataset out to use the defaults.

| Key | Meaning |
|---|---|
| `label` | the name the picker shows, e.g. `"Normal nerve"` (no sample IDs) |
| `group` | picker heading: `"proteomics"`, `"transcriptomics"` or `"scrnaseq"` (set automatically) |
| `src` | use files from another place instead of the `inputs/` subfolder (file or folder, same layout) |
| `pixel_size` | µm per full-resolution pixel, when the file does not carry it |
| `channels` | proteomics_file only: channel names in order, comma-separated |
| `crop` | `"x0,y0,x1,y1"` in full-resolution pixels: keep only that rectangle (not Xenium, CosMx or MERSCOPE) |
| `flip` | `"h"`, `"v"` or `"hv"`: mirror the image (not Xenium, CosMx or MERSCOPE) |
| `image_scale` | Visium only: your image width ÷ the width of the image Space Ranger used |
| `user_layer` | Visium only: the layer your own UMAP / clusters belong to, e.g. `"008um"`, `"016um"` or `"spots"` (needed when you give them) |
| `user_umap` | your embedding CSV (`barcode,UMAP_1,UMAP_2`); default `my_umap.csv` in the dataset folder |
| `user_obs` | your clusters CSV, or a list of them; default `my_clusters.csv` and `my_clusters_*.csv` in the dataset folder |
| `user_h5ad` | an `.h5ad` instead of the CSVs; default `my_analysis.h5ad` in the dataset folder |
| `user_embedding` | the `obsm` key of the embedding in `user_h5ad` (default `X_umap`) |
| `user_columns` | only these columns become clusterings, e.g. `"seurat_clusters,cell_type"` |
| `user_barcode_prefix` / `user_barcode_suffix` | a merged object with several sections: keep this section's barcodes (e.g. `"S1_"`, `"_2"`) |
| `user_replace` | `True`: show only your clusterings, not SODA's Leiden ones |
| `no_cells` | Xenium, CosMx, MERSCOPE: `True` skips the cells layer |
| `no_transcripts` | Xenium, CosMx, MERSCOPE: `True` skips the molecules (the slowest part of a Xenium prepare) |
| `qv_min` | Xenium only: lowest transcript quality value kept (default 20) |
| `images` | CosMx: `"morphology"`, `"composite"` or `"none"` instead of the automatic choice; MERSCOPE: `"none"` (a transcript-density background instead of the images) |
| `z` | MERSCOPE: the z-plane of the mosaic images to use (default: the middle one present); CosMx: the plane of raw z-stack images |
| `cells_dir` | MERSCOPE only: the folder holding `cell_by_gene.csv` and `cell_metadata.csv` when there are several segmentation runs |

In [ ]:
OPTIONS = {
    # "nerve_section1": {"label": "Normal nerve"},
    # "neuroma_panel":  {"label": "Painful neuroma", "pixel_size": 0.5},
    # "big_scan":       {"src": r"E:\scans\slide_07.qptiff", "label": "Whole slide"},
    # "hd_section2":    {"user_layer": "008um"},   # + my_umap.csv / my_clusters.csv in its folder
}

GROUP = {"proteomics_file": "proteomics", "proteomics_markers": "proteomics", "visium": "transcriptomics",
         "xenium": "transcriptomics", "cosmx": "transcriptomics", "merscope": "transcriptomics", "scrnaseq": "scrnaseq"}
IMG = (".tif", ".tiff", ".btf")

def _one(folder, patterns):
    hits = sorted({p for pat in patterns for p in folder.glob(pat) if p.is_file()})
    return hits

def check_dataset(kind, ds_id, src):
    # Returns (inputs for prepare.py, problems, notes)
    found, problems, notes = {}, [], []
    if kind == "proteomics_file":
        files = [src] if src.is_file() else _one(src, ["*.qptiff", "*.ome.tif", "*.ome.tiff", "*.tif", "*.tiff"])
        if len(files) == 1:
            found["src"] = files[0]
        elif not files:
            problems.append("no image file (*.qptiff / *.ome.tif / *.tif)")
        else:
            problems.append(f"{len(files)} image files; put exactly one: " + ", ".join(f.name for f in files[:4]))
    elif kind == "proteomics_markers":
        tifs = [f for f in _one(src, ["*.tif", "*.tiff"])]
        named = [f for f in tifs if f.stem.split("_")[0].isdigit()]
        if not tifs:
            problems.append("no marker TIFFs")
        elif len(named) < len(tifs):
            problems.append(f"{len(tifs) - len(named)} TIFF(s) not named NN_Marker_..._gray.tif")
        found["src"] = src
        notes.append(f"{len(tifs)} marker file(s)")
        if "pixel_size" not in OPTIONS.get(ds_id, {}):
            notes.append("no pixel_size option: no scale bar or protein squares unless the files carry it")
    elif kind == "visium":
        outs = src / "outs"
        images = [f for f in src.iterdir() if f.is_file() and f.suffix.lower() in IMG] if src.is_dir() else []
        if len(images) == 1:
            found["image"] = images[0]
        elif not images:
            problems.append("no full-resolution image (.tif/.tiff/.btf) next to outs/")
        else:
            problems.append(f"{len(images)} images; keep only the one given to spaceranger --image")
        if not outs.is_dir():
            problems.append("no outs/ folder")
        else:
            found["outs"] = outs
            binned = outs / "binned_outputs"
            if binned.is_dir():
                want = VISIUM_BINS.split(",") if VISIUM_BINS else [p.name.replace("square_", "") for p in binned.iterdir()]
                for b in want:
                    layer = binned / f"square_{b}"
                    for need in ("filtered_feature_bc_matrix.h5", "spatial/tissue_positions.parquet"):
                        if not (layer / need).exists():
                            problems.append(f"missing outs/binned_outputs/square_{b}/{need}")
                notes.append("Visium HD, layers " + ", ".join(want))
            else:
                if not (outs / "filtered_feature_bc_matrix.h5").exists():
                    problems.append("missing outs/filtered_feature_bc_matrix.h5 (or outs/binned_outputs/ for HD)")
                if not any((outs / "spatial" / n).exists() for n in
                           ("tissue_positions.parquet", "tissue_positions.csv", "tissue_positions_list.csv")):
                    problems.append("missing outs/spatial/tissue_positions.parquet (or .csv)")
                notes.append("classic Visium (55 um spots)")
        user_files(ds_id, src, found, problems, notes)
    elif kind == "xenium":
        stains = _one(src / "morphology_focus", ["*.ome.tif", "*.ome.tiff"])
        if not stains:
            problems.append("missing morphology_focus/*.ome.tif")
        found["src"] = src
        notes.append(f"{len(stains)} stain file(s)")
        if (src / "Custom_annotation.geojson").exists():
            notes.append("Custom_annotation.geojson -> overlay")
        cells = [n for n in ("cells.parquet", "cells.csv.gz", "cells.csv") if (src / n).exists()]
        matrix = [n for n in ("cell_feature_matrix.h5", "cell_feature_matrix") if (src / n).exists()]
        tx = [n for n in ("transcripts.parquet", "transcripts.zarr.zip", "transcripts.zarr") if (src / n).exists()]
        if cells and matrix:
            notes.append(f"cells: {cells[0]} + {matrix[0]}")
        else:
            notes.append("no cells layer (needs cells.parquet or cells.csv.gz AND cell_feature_matrix)")
        notes.append(f"molecules: {tx[0]}" if tx else "no transcripts file: no molecules")
        if any((src / n).exists() for n in ("cell_boundaries.parquet", "cell_boundaries.csv.gz")):
            notes.append("cell_boundaries: not used yet")
    elif kind == "cosmx":
        def flat(tag):                                # the flat files may sit up to two folders down
            hits = [p for pat in (f"*{tag}.csv", f"*{tag}.csv.gz", f"*{tag}.parquet")
                    for depth in ("", "*/", "*/*/") for p in src.glob(depth + pat)]
            return sorted(hits)
        ex, meta, fovs, tx = flat("exprMat_file"), flat("metadata_file"), flat("fov_positions_file"), flat("tx_file")
        found["src"] = src
        for name, hits in (("exprMat_file", ex), ("metadata_file", meta), ("fov_positions_file", fovs), ("tx_file", tx)):
            if len(hits) > 1:
                problems.append(f"{len(hits)} *{name} files: one slide per dataset folder")
        imgs = {d.name: len([f for f in d.iterdir() if f.is_file()]) for d in src.rglob("*")
                if d.is_dir() and d.name.lower() in ("morphology2d", "cellcomposite", "celllabels")}
        if not (ex or meta or tx or imgs):
            problems.append("no CosMx files (*_exprMat_file.csv, *_metadata_file.csv, *_tx_file.csv, CellComposite/ ...)")
        notes.append("images: " + (", ".join(f"{k} ({v} files)" for k, v in sorted(imgs.items())) if imgs
                                   else "none: a transcript-density background will be used"))
        if imgs and not (fovs or meta):
            problems.append("FOV images but no *_fov_positions_file.csv or *_metadata_file.csv to place them")
        notes.append(f"cells: {ex[0].name} + {meta[0].name}" if ex and meta
                     else "no cells layer (needs *_exprMat_file AND *_metadata_file)")
        notes.append(f"molecules: {tx[0].name}" if tx else "no *_tx_file: no molecules")
    elif kind == "merscope":
        found["src"] = src
        idir = src / "images"
        mos = sorted(idir.glob("mosaic_*_z*.tif*")) if idir.is_dir() else []
        if mos and not (idir / "micron_to_mosaic_pixel_transform.csv").exists():
            problems.append("images/ has mosaics but no micron_to_mosaic_pixel_transform.csv")
        planes = sorted({f.stem.rsplit("_z", 1)[-1] for f in mos})
        notes.append(f"images: {len(mos)} mosaic file(s), z-planes {', '.join(planes)}" if mos
                     else "no images/mosaic_*.tif: a transcript-density background will be used")
        def table(stem):
            hits = [p for pat in (f"{stem}.csv", f"{stem}.csv.gz", f"{stem}.parquet")
                    for depth in ("", "*/") for p in src.glob(depth + pat)]
            return sorted(hits, key=lambda p: len(p.parts))
        cbg, cmeta, tx = table("cell_by_gene"), table("cell_metadata"), table("detected_transcripts")
        if not (mos or cbg or tx):
            problems.append("no MERSCOPE files (images/mosaic_*.tif, cell_by_gene.csv, detected_transcripts.csv)")
        notes.append(f"cells: {cbg[0].relative_to(src).as_posix()} + {cmeta[0].relative_to(src).as_posix()}" if cbg and cmeta
                     else "no cells layer (needs cell_by_gene.csv AND cell_metadata.csv)")
        notes.append(f"molecules: {tx[0].name}" if tx else "no detected_transcripts file: no molecules")
    elif kind == "scrnaseq":
        h5ads = [src] if src.is_file() and src.suffix == ".h5ad" else _one(src, ["*.h5ad"])
        mtx = src / "filtered_feature_bc_matrix"
        if len(h5ads) == 1:
            found["h5ad"] = h5ads[0]
        elif len(h5ads) > 1:
            problems.append(f"{len(h5ads)} .h5ad files; put exactly one")
        elif mtx.is_dir():
            found["mtx"] = mtx
            umap = src / "analysis" / "umap" / "gene_expression_2_components" / "projection.csv"
            umap = umap if umap.exists() else src / "umap.csv"
            if umap.exists():
                found["umap"] = umap
            else:
                problems.append("missing the UMAP: analysis/umap/gene_expression_2_components/projection.csv or umap.csv")
            clus = src / "analysis" / "clustering" / "gene_expression_graphclust" / "clusters.csv"
            obs = ([clus] if clus.exists() else []) + [f for f in _one(src, ["*.csv"]) if f.name != "umap.csv"]
            found["obs"] = obs
            notes.append(f"10x matrix + UMAP, {len(obs)} colour-by file(s)")
        else:
            problems.append("no .h5ad and no filtered_feature_bc_matrix/ folder")
    return found, problems, notes

def user_files(ds_id, src, found, problems, notes):
    # Your own UMAP / clusters (Section 1): OPTIONS user_* keys, else my_umap.csv, my_clusters*.csv,
    # my_analysis.h5ad in the dataset folder
    opt = OPTIONS.get(ds_id, {})
    here = src if src.is_dir() else src.parent
    def path(p):
        p = Path(p)
        return p if p.is_absolute() else here / p
    umap = path(opt["user_umap"]) if "user_umap" in opt else here / "my_umap.csv"
    obs = opt.get("user_obs")
    obs = ([path(p) for p in ([obs] if isinstance(obs, (str, Path)) else obs)] if obs is not None
           else [here / "my_clusters.csv"] + sorted(here.glob("my_clusters_*.csv")))
    h5ad = path(opt["user_h5ad"]) if "user_h5ad" in opt else here / "my_analysis.h5ad"
    u = {"umap": umap if umap.exists() else None, "obs": [p for p in obs if p.exists()],
         "h5ad": h5ad if h5ad.exists() else None}
    for k, p in (("user_umap", umap), ("user_h5ad", h5ad)):
        if k in opt and not p.exists():
            problems.append(f"{k}: not found: {p}")
    if "user_obs" in opt:
        problems += [f"user_obs: not found: {p}" for p in obs if not p.exists()]
    if not (u["umap"] or u["obs"] or u["h5ad"]):
        return
    if u["h5ad"] and (u["umap"] or u["obs"]):
        problems.append("your own results: give either the .h5ad or the CSVs, not both")
    if not opt.get("user_layer"):
        problems.append("your own UMAP / clusters found: set user_layer (e.g. \"008um\") in OPTIONS")
    found["user"] = u
    names = [p.name for p in [u["umap"], u["h5ad"], *u["obs"]] if p]
    notes.append(f"your own results for layer {opt.get('user_layer')}: " + ", ".join(names))

def find_datasets():
    rows = []
    for kind in KINDS:
        for sub in sorted(p for p in (INPUTS / kind).iterdir() if p.is_dir()):
            rows.append((kind, sub.name, sub))
    have = {r[1] for r in rows}
    for ds_id, opt in OPTIONS.items():           # datasets that only exist as a `src` somewhere else
        if "src" in opt and ds_id not in have:
            kind = opt.get("kind")
            if kind not in KINDS:
                print(f"{ds_id}: give 'kind' (one of {list(KINDS)}) when the dataset has no inputs/ folder")
                continue
            rows.append((kind, ds_id, None))
    out = []
    for kind, ds_id, folder in rows:
        opt = OPTIONS.get(ds_id, {})
        src = Path(opt["src"]) if "src" in opt else folder
        if src is None or not src.exists():
            out.append(dict(id=ds_id, kind=kind, src=src, found={}, problems=[f"not found: {src}"], notes=[]))
            continue
        found, problems, notes = check_dataset(kind, ds_id, src)
        bad = [c for c in ds_id if not (c.isalnum() or c in "_-")]
        if bad:
            problems.append("dataset id may only use letters, digits, _ and -")
        out.append(dict(id=ds_id, kind=kind, src=src, found=found, problems=problems, notes=notes))
    return out

DATASETS = find_datasets()
if not DATASETS:
    print(f"No datasets yet. Put them under {INPUTS} as described in Section 1, then run this cell again.")
for d in DATASETS:
    status = "READY" if not d["problems"] else "FIX  "
    print(f"{status} {d['kind']:19s} {d['id']}")
    for p in d["problems"]:
        print(f"        problem: {p}")
    for n in d["notes"]:
        print(f"        note:    {n}")
READY = [d for d in DATASETS if not d["problems"]]
print(f"\n{len(READY)} of {len(DATASETS)} dataset(s) ready.")

Optional: look inside a proteomics or H&E file before building (axes, size, levels, channel
names). Nothing is written.

In [ ]:
for d in READY:
    if d["kind"] == "proteomics_file":
        subprocess.run([PY, str(SV / "prepare.py"), "inspect", "--src", str(d["found"]["src"])])

## 3. Prepare: read each dataset once into `cache/`

This is the slow step: it reads the full image once and builds a multi-resolution copy plus a
gene-by-bin index. The cache holds the full-resolution raw image, so **it stays on this
computer and is never uploaded**. A dataset with `cache/<id>/manifest.json` is skipped; set
`REBUILD = True` to redo it (then also delete `export/<id>/`, or old tiles stay).

Single-cell RNA-seq has no cache: it is written straight to `export/` here.

In [ ]:
REBUILD = False

def run(args, logfile=None):
    # Run one script, print its output as it comes, keep a log. Returns True on success.
    print("$", " ".join(str(a) for a in args))
    t0 = time.time()
    log = open(logfile, "a", encoding="utf-8") if logfile else None
    p = subprocess.Popen([str(a) for a in args], cwd=str(REPO), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout:
        print(line, end="")
        if log:
            log.write(line)
    p.wait()
    if log:
        log.close()
    print(f"-> exit {p.returncode} in {time.time() - t0:.0f} s\n")
    return p.returncode == 0

def prepare_args(d):
    o, f = OPTIONS.get(d["id"], {}), d["found"]
    common = ["--id", d["id"], "--out", CACHE, "--label", o.get("label", d["id"])]
    if o.get("pixel_size"):
        common += ["--pixel-size", o["pixel_size"]]
    k = d["kind"]
    if k == "proteomics_file":
        a = ["tiff", "--src", f["src"]] + (["--channels", o["channels"]] if o.get("channels") else [])
    elif k == "proteomics_markers":
        a = ["akoya", "--src", f["src"]]
    elif k == "visium":
        a = ["visium", "--image", f["image"], "--outs", f["outs"]]
        if (f["outs"] / "binned_outputs").is_dir() and VISIUM_BINS:
            a += ["--bins", VISIUM_BINS]
        if o.get("image_scale"):
            a += ["--image-scale", o["image_scale"]]
    elif k == "xenium":
        a = ["xenium", "--src", f["src"]]
        a += (["--no-cells"] if o.get("no_cells") else []) + (["--no-transcripts"] if o.get("no_transcripts") else [])
        a += (["--qv-min", o["qv_min"]] if o.get("qv_min") is not None else [])
    elif k in ("cosmx", "merscope"):
        a = [k, "--src", f["src"]]
        a += (["--no-cells"] if o.get("no_cells") else []) + (["--no-transcripts"] if o.get("no_transcripts") else [])
        a += (["--images", o["images"]] if o.get("images") else []) + (["--z", o["z"]] if o.get("z") is not None else [])
        a += (["--cells-dir", o["cells_dir"]] if k == "merscope" and o.get("cells_dir") else [])
    if k not in ("xenium", "cosmx", "merscope"):
        a += (["--crop", o["crop"]] if o.get("crop") else []) + (["--flip", o["flip"]] if o.get("flip") else [])
    return [PY, SV / "prepare.py"] + a + common

def rnaseq_args(d):
    o, f = OPTIONS.get(d["id"], {}), d["found"]
    a = [PY, SV / "prepare.py", "rnaseq", "--id", d["id"], "--label", o.get("label", d["id"]), "--dest", EXPORT]
    if "h5ad" in f:
        a += ["--h5ad", f["h5ad"]]
    else:
        a += ["--mtx", f["mtx"], "--umap", f["umap"]]
        for p in f["obs"]:
            a += ["--obs", p]
    return a

(WORK / "logs").mkdir(exist_ok=True)
results = {}
for d in READY:
    log = WORK / "logs" / f"{d['id']}_prepare.log"
    if d["kind"] == "scrnaseq":
        if (EXPORT / d["id"] / "manifest.json").exists() and not REBUILD:
            print(f"skip {d['id']}: already in export/"); results[d["id"]] = True; continue
        results[d["id"]] = run(rnaseq_args(d), log)
    else:
        if (CACHE / d["id"] / "manifest.json").exists() and not REBUILD:
            print(f"skip {d['id']}: already in cache/"); results[d["id"]] = True; continue
        results[d["id"]] = run(prepare_args(d), log)
print("prepared:", [k for k, v in results.items() if v])
print("FAILED:  ", [k for k, v in results.items() if not v] or "none")

## 4. Export: write the upload-ready files into `export/`

For every prepared image dataset this writes, into `export/<id>/`:

- `tiles/`: the image cut into 1024 x 1024 pieces at every zoom level, one channel per file;
- `expr/` (Visium): one file per gene per bin size, the bin positions, the whole-section UMAP
  with Leiden clusters (plus your own UMAP / clusters when you gave them, matched by barcode:
  the log says how many of your barcodes were found), and the data the in-browser circle -> UMAP
  tool reads;
- `expr/` (proteomics with a pixel size): the mean of each marker per 16 µm square, with a UMAP;
- `manifest.json`: the index the viewer reads (without your local file paths).

Re-running skips tiles and gene files that already exist. Check the log for
`[export] verified N channel(s)` and no line starting with `***`.

In [ ]:
exported = {}
for d in READY:
    if d["kind"] == "scrnaseq" or not results.get(d["id"]):
        continue
    a = [PY, SV / "prepare.py", "export", "--id", d["id"], "--out", CACHE, "--dest", EXPORT,
         "--fmt", TILE_FORMAT, "--workers", WORKERS]
    group = "visium" if d["kind"] == "visium" else "proteomics"
    if MARKERS.get(group):
        a += ["--markers", MARKERS[group]]
    u, opt = d["found"].get("user"), OPTIONS.get(d["id"], {})
    if u:                                        # your own UMAP / clusters (Section 1)
        a += ["--user-layer", opt["user_layer"]]
        a += ["--user-umap", u["umap"]] if u["umap"] else []
        for p in u["obs"]:
            a += ["--user-obs", p]
        a += ["--user-h5ad", u["h5ad"]] if u["h5ad"] else []
        for k in ("user_embedding", "user_columns", "user_barcode_prefix", "user_barcode_suffix"):
            if opt.get(k):
                a += ["--" + k.replace("_", "-"), opt[k]]
        a += ["--user-replace"] if opt.get("user_replace") else []
    exported[d["id"]] = run(a, WORK / "logs" / f"{d['id']}_export.log")
    txt = (WORK / "logs" / f"{d['id']}_export.log").read_text(encoding="utf-8", errors="replace")
    if "***" in txt:
        print(f"WARNING {d['id']}: the export log has a *** line; read it before uploading.")
print("exported:", [k for k, v in exported.items() if v])
print("FAILED:  ", [k for k, v in exported.items() if not v] or "none")

## 5. De-identification check

Scans every file name and every text value in `export/` for anything that looks like a sample
ID. It must end with **`[deid] clean`**. If it lists matches: rename the source (folder name,
label, channel or column name), delete `export/<id>/`, and run Sections 3-5 again. A gene name
can match by accident, so read the list before renaming anything.

Also read the labels and channel names yourself: the check only knows the ID patterns set in
`deid.py` (other labs: edit `_ID` there first, see `docs/DEIDENTIFICATION.md`).

In [ ]:
r = subprocess.run([PY, str(SV / "deid.py"), "check", str(EXPORT)], cwd=str(REPO),
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-5000:] or r.stderr[-2000:])
DEID_CLEAN = r.returncode == 0 and "[deid] clean" in r.stdout
print("\nOK to upload." if DEID_CLEAN else "\nNOT clean: fix the listed names before uploading.")

## 6. Look before you upload

The size of each dataset (many hosts limit file count and total size), then the exact
`export/` folder opened in your browser on this computer. Nothing leaves your computer here.

In [ ]:
def folder_size(p):
    n = b = 0
    for f in p.rglob("*"):
        if f.is_file():
            n += 1; b += f.stat().st_size
    return n, b

tot_n = tot_b = 0
for ds in sorted(p for p in EXPORT.iterdir() if p.is_dir()):
    n, b = folder_size(ds)
    tot_n += n; tot_b += b
    print(f"{ds.name:40s} {n:>9,d} files {b / 1e9:8.2f} GB")
print(f"{'TOTAL':40s} {tot_n:>9,d} files {tot_b / 1e9:8.2f} GB")

In [ ]:
# Starts two small local servers: one for the export folder, one for the viewer page.
import socket
def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]
DATA_PORT, PAGE_PORT = free_port(), free_port()
servers = [
    subprocess.Popen([PY, str(SV / "static_serve.py"), str(EXPORT), str(DATA_PORT)], cwd=str(SV)),
    subprocess.Popen([PY, "-m", "http.server", str(PAGE_PORT), "--bind", "127.0.0.1", "--directory", str(SV)]),
]
time.sleep(1.5)
for ds in sorted(p.name for p in EXPORT.iterdir() if p.is_dir()):
    print(f"http://127.0.0.1:{PAGE_PORT}/index.html?static=http://127.0.0.1:{DATA_PORT}&ds={ds}")
print("\nOpen a link, click through every dataset, then run the next cell to stop the servers.")

In [ ]:
for s in servers:
    s.terminate()
print("servers stopped")

## 7. Upload and link

1. Make one folder in the cloud for SODA (on Azure: a blob container, e.g. `spatial`).
2. Copy that folder's link and paste it into `CLOUD_URL` below. Paste only the address
   (e.g. `https://myaccount.blob.core.windows.net/spatial`), never a key or a `?sv=...` pass.
3. Run the cell. It prints the upload command and the lines for `config.js`. **It runs nothing.**
4. Upload **the whole `export/` folder as it is** (one subfolder per dataset), then paste the
   printed lines into `web/SpatialViewer/config.js`. Never upload `cache/`, `inputs/` or the key file.

`MODE` picks who can open the data (see `docs/HOSTING.md`):

- **`"private"`**: the data stays in a private container and readers need the password
  (the page's `/api/token` hands out a read-only pass for 1 hour).
- **`"public"`**: anyone with the link can open (and download) the data. Good for examples and
  published data; you pay for what people download, so keep public sets small.


In [ ]:
MODE = "private"                              # "private" (password) or "public" (anyone with the link)
CLOUD_URL = "<paste your cloud folder link>"  # EDIT: e.g. https://myaccount.blob.core.windows.net/spatial

if not DEID_CLEAN:
    print("Run Section 5 until it says clean before uploading.\n")

url = CLOUD_URL.strip()
if "?" in url:                              # a pasted pass (SAS) is a secret: drop it, never print it
    url = url.split("?", 1)[0]
    print("Note: removed the '?...' part of CLOUD_URL. Paste only the folder address here.\n")
base_url = url.rstrip("/")
if not base_url.startswith(("https://", "http://")) or "<" in base_url:
    raise ValueError("Set CLOUD_URL to your cloud folder's link, e.g. https://myaccount.blob.core.windows.net/spatial")
on_azure = ".blob.core.windows.net/" in base_url + "/"

lines = []
for ds in sorted(p for p in EXPORT.iterdir() if p.is_dir()):
    man = json.loads((ds / "manifest.json").read_text(encoding="utf-8"))
    d = next((x for x in DATASETS if x["id"] == ds.name), None)
    o = OPTIONS.get(ds.name, {})
    group = o.get("group") or (GROUP[d["kind"]] if d else "transcriptomics")
    label = o.get("label") or man.get("label") or ds.name
    entry = f'{ds.name}: {{ group: "{group}", label: {json.dumps(label)}'
    entry += f', base: "{base_url}/{ds.name}/" }},' if MODE == "public" else " },"
    lines.append("    " + entry)

print("# 1. Upload export/ to", base_url)
if on_azure:
    print("# Make a short-lived WRITE pass (SAS) for the container first")
    print("# (Azure portal or Storage Explorer: container > Generate SAS, permissions Write + Create + List).")
    print('$sas = "<paste the SAS token here, starting with sv=>"')
    print(f'azcopy copy "{EXPORT}\\*" "{base_url}?$sas" --recursive')
else:
    print(f"# Copy everything inside {EXPORT} to that folder with your host's upload tool.")
if MODE == "public":
    print("# Public data: allow anonymous read and a CORS rule for GET/HEAD from the page's address")
    print("# (Azure: deploy/azure/README.md, option A).")

print("\n// 2. Paste into web/SpatialViewer/config.js (copy from config.example.js first)")
print("datasets: {\n" + "\n".join(lines) + "\n},")
print('tokenEndpoint: "/api/token",' if MODE == "private" else 'tokenEndpoint: "",')


### Before you share the link

- [ ] Section 5 says `[deid] clean`, and you read the labels and channel names yourself.
- [ ] Every dataset opened and looked right in Section 6.
- [ ] Only `export/` was uploaded: no `cache/`, raw files or key file.
- [ ] `config.js` lists every dataset, with labels that name the tissue and condition only.
- [ ] Private data: the password works and a link without it shows nothing.